# 08 - Consolidacion del Ground Truth manual de RUCs

Este notebook consolida el diccionario manual de alias empresa -> RUC -> razon social construido a partir de busquedas verificadas en fuentes oficiales.

La etapa automatizada de Entity Resolution se mantiene como baseline metodologico y propuesta de productizacion. Para el modelo final, este notebook genera el Golden Record manual que sera usado por Feature Engineering.

## Objetivo

- Leer todos los archivos Excel/CSV de `02_data_cleaning/data_ruc_universo_empresas/new/`.
- Homologar encabezados equivalentes entre archivos.
- Limpiar `nombre_original`, `razon_social` y `ruc`.
- Filtrar registros con RUC ecuatoriano de 13 digitos, nombre original y razon social valida.
- Usar todos los RUC verificados disponibles en `leads_ruc_new.xlsx` y `proyectos_empresa_ruc_new.xlsx`, incluso cuando el pais comercial/origen no sea Ecuador.
- Registrar motivos de descarte para auditar RUC invalido, razon social faltante o nombre faltante.
- Deduplicar a nivel de alias manual verificado.
- Exportar `match_final_empresas_verificado.csv` para ser consumido por `03_feature_engineering/01_base_sri.ipynb`.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

In [2]:
def find_project_root(start: Path) -> Path:
    """Find the repository root from either the repo root or a notebook subfolder."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        expected = candidate / "02_data_cleaning" / "data_ruc_universo_empresas" / "new"
        if expected.exists():
            return candidate
    raise FileNotFoundError("No se encontro la carpeta 02_data_cleaning/data_ruc_universo_empresas/new")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "02_data_cleaning" / "data_ruc_universo_empresas"
INPUT_DIR = DATA_DIR / "new"

OUTPUT_PATH = DATA_DIR / "match_final_empresas_verificado.csv"
DISCARDED_PATH = DATA_DIR / "match_final_empresas_verificado_descartados.csv"
CONFLICT_ALIAS_PATH = DATA_DIR / "match_final_empresas_verificado_conflictos_alias.csv"
CONFLICT_RUC_PATH = DATA_DIR / "match_final_empresas_verificado_conflictos_ruc.csv"

SUPPORTED_SUFFIXES = {".xlsx", ".xls", ".csv"}
source_files = sorted(path for path in INPUT_DIR.iterdir() if path.suffix.lower() in SUPPORTED_SUFFIXES)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input dir: {INPUT_DIR}")
print(f"Archivos encontrados: {len(source_files)}")
for path in source_files:
    print(f"- {path.name}")

if not source_files:
    raise FileNotFoundError(f"No se encontraron archivos Excel/CSV en {INPUT_DIR}")

Project root: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria
Input dir: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_ruc_universo_empresas\new
Archivos encontrados: 2
- leads_ruc_new.xlsx
- proyectos_empresa_ruc_new.xlsx


## Funciones de limpieza y homologacion

Los archivos manuales pueden tener encabezados equivalentes con diferencias menores, por ejemplo `RUC ECUADOR` vs `RUC`, o `Sede en ECUADOR` vs `Sede en Ecuador`. Por eso se normalizan los nombres de columnas antes de extraer los campos canonicos.

In [3]:
def strip_accents(text: str) -> str:
    return "".join(
        char
        for char in unicodedata.normalize("NFD", text)
        if unicodedata.category(char) != "Mn"
    )


def normalize_column_name(value) -> str:
    text = "" if pd.isna(value) else str(value)
    text = strip_accents(text).lower().strip()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().strip('"').strip("'").strip()
    text = re.sub(r"\s+", " ", text)
    if not text or text.lower() in {"nan", "none", "null", "na", "n/a"}:
        return pd.NA
    return text


def normalize_entity(value):
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    text = strip_accents(str(text)).upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_ruc(value):
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA
    digits = re.sub(r"\D+", "", str(text))
    return digits if digits else pd.NA


def clean_sede_ecuador(value):
    text = clean_text(value)
    if pd.isna(text):
        return pd.NA

    numeric_candidate = str(text).replace(",", ".")
    try:
        number = float(numeric_candidate)
        if number == 1:
            return True
        if number == 0:
            return False
    except ValueError:
        pass

    normalized = normalize_column_name(text)
    if normalized in {"true", "si", "s", "yes", "y", "ecuador", "ec"}:
        return True
    if normalized in {"false", "no", "n"}:
        return False
    return pd.NA


def normalize_country(value):
    text = normalize_entity(value)
    if pd.isna(text):
        return pd.NA
    if text in {"EC", "ECUADOR", "REPUBLICA DEL ECUADOR"}:
        return "ECUADOR"
    return text


def first_non_empty(values):
    for value in values:
        if not pd.isna(value) and str(value).strip():
            return value
    return pd.NA


def join_unique(values):
    uniques = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in uniques:
            uniques.append(text)
    return " | ".join(sorted(uniques))


def detect_columns(df: pd.DataFrame) -> dict:
    normalized = {column: normalize_column_name(column) for column in df.columns}

    empresa_col = next(
        (
            column
            for column, name in normalized.items()
            if name == "empresa"
            or (
                "empresa" in name
                and "pais" not in name
                and "razon" not in name
                and "social" not in name
                and "ruc" not in name
            )
        ),
        None,
    )
    razon_col = next((column for column, name in normalized.items() if "razon social" in name), None)
    ruc_col = next((column for column, name in normalized.items() if "ruc" in name), None)
    pais_col = next((column for column, name in normalized.items() if "pais" in name), None)
    sede_col = next((column for column, name in normalized.items() if "sede" in name), None)

    missing = []
    if empresa_col is None:
        missing.append("empresa")
    if razon_col is None:
        missing.append("razon_social")
    if ruc_col is None:
        missing.append("ruc")
    if missing:
        raise ValueError(f"No se pudieron detectar columnas requeridas: {missing}. Columnas disponibles: {list(df.columns)}")

    return {
        "empresa": empresa_col,
        "razon_social": razon_col,
        "ruc": ruc_col,
        "pais_empresa": pais_col,
        "sede_ecuador": sede_col,
    }

In [4]:
def iter_source_tables(path: Path):
    suffix = path.suffix.lower()

    if suffix in {".xlsx", ".xls"}:
        workbook = pd.ExcelFile(path)
        for sheet_name in workbook.sheet_names:
            yield sheet_name, pd.read_excel(path, sheet_name=sheet_name, dtype=str)
        return

    if suffix == ".csv":
        last_error = None
        for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
            try:
                df = pd.read_csv(path, dtype=str, sep=None, engine="python", encoding=encoding)
                yield path.stem, df
                return
            except UnicodeDecodeError as exc:
                last_error = exc
        raise last_error

    raise ValueError(f"Extension no soportada: {path.suffix}")


raw_frames = []
schema_log = []

for path in source_files:
    for sheet_name, df in iter_source_tables(path):
        columns = detect_columns(df)
        schema_log.append(
            {
                "archivo_fuente": path.name,
                "hoja_fuente": sheet_name,
                "filas": len(df),
                "columna_nombre_original": columns["empresa"],
                "columna_razon_social": columns["razon_social"],
                "columna_ruc": columns["ruc"],
                "columna_pais": columns["pais_empresa"],
                "columna_sede": columns["sede_ecuador"],
            }
        )

        frame = pd.DataFrame(
            {
                "nombre_original": df[columns["empresa"]].map(clean_text),
                "razon_social": df[columns["razon_social"]].map(clean_text),
                "ruc": df[columns["ruc"]].map(clean_ruc),
                "pais_empresa": df[columns["pais_empresa"]].map(clean_text) if columns["pais_empresa"] else pd.NA,
                "sede_ecuador": df[columns["sede_ecuador"]].map(clean_sede_ecuador) if columns["sede_ecuador"] else pd.NA,
                "archivo_fuente": path.name,
                "hoja_fuente": sheet_name,
                "fila_fuente": df.index + 2,
            }
        )
        raw_frames.append(frame)

raw = pd.concat(raw_frames, ignore_index=True)
schema_log = pd.DataFrame(schema_log)

raw["nombre_original_norm"] = raw["nombre_original"].map(normalize_entity)
raw["razon_social_norm"] = raw["razon_social"].map(normalize_entity)
raw["ruc_valido_13_digitos"] = raw["ruc"].astype("string").str.fullmatch(r"\d{13}").fillna(False)
raw["pais_empresa_norm"] = raw["pais_empresa"].map(normalize_country)
raw["fuente_es_leads_new"] = raw["archivo_fuente"].astype(str).str.contains("leads_ruc_new", case=False, regex=False)
raw["fuente_es_proyectos_new"] = raw["archivo_fuente"].astype(str).str.contains("proyectos_empresa_ruc_new", case=False, regex=False)

print(f"Filas crudas leidas: {len(raw):,}")
display(schema_log)

Filas crudas leidas: 853


,archivo_fuente,hoja_fuente,filas,columna_nombre_original,columna_razon_social,columna_ruc,columna_pais,columna_sede
0,leads_ruc_new.xlsx,Sheet0,441,Empresa,Nueva Razon Social Empresa en ECUADOR,RUC ECUADOR,Pais de la Empresa,Sede en ECUADOR
1,proyectos_empresa_ruc_new.xlsx,Hoja 1,412,Empresa,Nueva Razon Social Empresa en Ecuador,RUC,Pais de la Empresa,Sede en Ecuador


## Consolidacion del Golden Record

El archivo maestro se conserva a nivel de alias, no solo a nivel de RUC. Esto es importante porque el CRM trabaja con nombres comerciales distintos que pueden apuntar a la misma razon social oficial.

Regla de alcance actual: se usan todos los RUC ecuatorianos verificados manualmente disponibles en `leads_ruc_new.xlsx` y `proyectos_empresa_ruc_new.xlsx`. El pais comercial/origen queda como trazabilidad, pero no excluye un registro si existe razon social y RUC ecuatoriano validado.

In [5]:
def motivo_descarte(row):
    motivos = []
    if not bool(row["ruc_valido_13_digitos"]):
        motivos.append("ruc_invalido_o_vacio")
    if pd.isna(row["nombre_original"]):
        motivos.append("nombre_original_vacio")
    if pd.isna(row["razon_social"]):
        motivos.append("razon_social_vacia")
    return " | ".join(motivos) if motivos else "VALIDO"


valid_mask = (
    raw["ruc_valido_13_digitos"]
    & raw["nombre_original"].notna()
    & raw["razon_social"].notna()
)
raw["motivo_descarte"] = raw.apply(motivo_descarte, axis=1)

valid = raw.loc[valid_mask].copy()
discarded = raw.loc[~valid_mask].copy()

valid["fuente_detalle"] = (
    valid["archivo_fuente"].astype(str)
    + "::"
    + valid["hoja_fuente"].astype(str)
    + "#fila_"
    + valid["fila_fuente"].astype(str)
)

master = (
    valid.sort_values(["nombre_original_norm", "ruc", "razon_social_norm", "archivo_fuente", "fila_fuente"])
    .groupby(["nombre_original_norm", "ruc", "razon_social_norm"], as_index=False, dropna=False)
    .agg(
        nombre_original=("nombre_original", first_non_empty),
        razon_social=("razon_social", first_non_empty),
        pais_empresa=("pais_empresa", join_unique),
        sede_ecuador=("sede_ecuador", first_non_empty),
        n_apariciones=("ruc", "size"),
        fuentes=("fuente_detalle", join_unique),
    )
)

master = master[
    [
        "nombre_original",
        "nombre_original_norm",
        "razon_social",
        "razon_social_norm",
        "ruc",
        "pais_empresa",
        "sede_ecuador",
        "n_apariciones",
        "fuentes",
    ]
].sort_values(["nombre_original_norm", "ruc"]).reset_index(drop=True)

print(f"Filas con RUC valido antes de deduplicar: {len(valid):,}")
print(f"Alias verificados unicos en master: {len(master):,}")
print(f"RUCs unicos en master: {master['ruc'].nunique():,}")
print(f"Filas descartadas para auditoria: {len(discarded):,}")
print("\nMotivos de descarte:")
print(discarded["motivo_descarte"].value_counts().to_string())
print("\nDistribucion de leads validos por pais normalizado:")
print(valid.loc[valid["fuente_es_leads_new"], "pais_empresa_norm"].value_counts(dropna=False).to_string())

display(master.head(10))

Filas con RUC valido antes de deduplicar: 541
Alias verificados unicos en master: 209
RUCs unicos en master: 181
Filas descartadas para auditoria: 312

Motivos de descarte:
motivo_descarte
ruc_invalido_o_vacio | razon_social_vacia    312

Distribucion de leads validos por pais normalizado:
pais_empresa_norm
EEUU             42
MEXICO           38
ECUADOR          29
COLOMBIA         10
ALEMANIA          8
REINO UNIDO       8
FRANCIA           6
BELGICA           4
SUIZA             4
INGLATERRA        3
PERU              3
ITALIA            2
ARGENTINA         2
BRASIL            2
CANADA            1
SUECIA            1
LUXEMBURGO        1
CHILE             1
PAISES BAJOS      1
COREA DEL SUR     1
TAIWAN            1
ESPANA            1


,nombre_original,nombre_original_norm,razon_social,razon_social_norm,ruc,pais_empresa,sede_ecuador,n_apariciones,fuentes
0,ABBOTT,ABBOTT,ABBOTT LABORATORIOS DEL ECUADOR CIA LTDA,ABBOTT LABORATORIOS DEL ECUADOR CIA LTDA,0990000670001,EEUU,True,1,leads_ruc_new.xlsx::Sheet0#fila_4
1,Abbvie,ABBVIE,ABBVIE S.A.S.,ABBVIE S A S,1792489156001,EEUU,True,1,leads_ruc_new.xlsx::Sheet0#fila_5
2,Adium,ADIUM,MEDICAMENTA ECUATORIANA S.A.,MEDICAMENTA ECUATORIANA S A,1790775941001,MEXICO,True,8,proyectos_empresa_ruc_new.xlsx::Hoja 1#fila_10 | proyectos_empresa_ruc_new.xlsx::Hoja 1#fila_3 | proyectos_empresa_r...
3,AIG,AIG,AIG METROPOLITANA CIA. DE SEGUROS Y REASEGUROS S.A.,AIG METROPOLITANA CIA DE SEGUROS Y REASEGUROS S A,1790475247001,ECUADOR,True,4,leads_ruc_new.xlsx::Sheet0#fila_11 | leads_ruc_new.xlsx::Sheet0#fila_12 | leads_ruc_new.xlsx::Sheet0#fila_13 | leads...
4,Akros,AKROS,AKROS CIA. LTDA.,AKROS CIA LTDA,1791148800001,ECUADOR,True,1,leads_ruc_new.xlsx::Sheet0#fila_15
5,AlimentosAlConsumidor,ALIMENTOSALCONSUMIDOR,CORPORACION DISTRIBUIDORA DE ALIMENTOS CORDIALSA S.A.S.,CORPORACION DISTRIBUIDORA DE ALIMENTOS CORDIALSA S A S,1791287169001,COLOMBIA,True,1,leads_ruc_new.xlsx::Sheet0#fila_17
6,Alpina,ALPINA,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR S.A.,ALPINA PRODUCTOS ALIMENTICIOS ALPIECUADOR S A,1791302400001,COLOMBIA,True,1,leads_ruc_new.xlsx::Sheet0#fila_21
7,Alvarez Barba S.A,ALVAREZ BARBA S A,Alvarez Barba S.A,ALVAREZ BARBA S A,1790360741001,ECUADOR,True,1,leads_ruc_new.xlsx::Sheet0#fila_22
8,AMBROSIA,AMBROSIA,AMBROSIA WINESHOP S.A.S.,AMBROSIA WINESHOP S A S,1793141250001,ECUADOR,True,1,leads_ruc_new.xlsx::Sheet0#fila_23
9,ANHEUSER-BUSCH INBEV,ANHEUSER BUSCH INBEV,CERVECERIA NACIONAL CN S.A.,CERVECERIA NACIONAL CN S A,0990023549001,BELGICA,True,3,leads_ruc_new.xlsx::Sheet0#fila_24 | leads_ruc_new.xlsx::Sheet0#fila_25 | leads_ruc_new.xlsx::Sheet0#fila_26


## Validaciones de consistencia

Estas validaciones detectan casos que requieren revision manual antes de conectar el Golden Record con SRI:

- Un mismo alias comercial asignado a mas de un RUC.
- Un mismo RUC asignado a mas de una razon social normalizada.

In [6]:
conflicts_alias = (
    master.groupby("nombre_original_norm", as_index=False)
    .agg(
        n_rucs=("ruc", "nunique"),
        rucs=("ruc", join_unique),
        nombres_originales=("nombre_original", join_unique),
        razones_sociales=("razon_social", join_unique),
    )
    .query("n_rucs > 1")
    .sort_values(["n_rucs", "nombre_original_norm"], ascending=[False, True])
)

conflicts_ruc = (
    master.groupby("ruc", as_index=False)
    .agg(
        n_razones=("razon_social_norm", "nunique"),
        razones_sociales=("razon_social", join_unique),
        nombres_originales=("nombre_original", join_unique),
    )
    .query("n_razones > 1")
    .sort_values(["n_razones", "ruc"], ascending=[False, True])
)

summary = pd.DataFrame(
    [
        {"metrica": "archivos_fuente", "valor": len(source_files)},
        {"metrica": "filas_crudas", "valor": len(raw)},
        {"metrica": "filas_validas_pre_deduplicacion", "valor": len(valid)},
        {"metrica": "alias_verificados_unicos", "valor": len(master)},
        {"metrica": "rucs_unicos", "valor": master["ruc"].nunique()},
        {"metrica": "filas_descartadas", "valor": len(discarded)},
        {"metrica": "conflictos_alias_multi_ruc", "valor": len(conflicts_alias)},
        {"metrica": "conflictos_ruc_multi_razon", "valor": len(conflicts_ruc)},
    ]
)

display(summary)

if not conflicts_alias.empty:
    print("Revision requerida: hay alias comerciales con mas de un RUC.")
    display(conflicts_alias)

if not conflicts_ruc.empty:
    print("Revision requerida: hay RUCs con mas de una razon social normalizada.")
    display(conflicts_ruc)

if conflicts_alias.empty and conflicts_ruc.empty:
    print("Validacion OK: no se detectaron conflictos estructurales en el Golden Record.")

,metrica,valor
0,archivos_fuente,2
1,filas_crudas,853
2,filas_validas_pre_deduplicacion,541
3,alias_verificados_unicos,209
4,rucs_unicos,181
5,filas_descartadas,312
6,conflictos_alias_multi_ruc,0
7,conflictos_ruc_multi_razon,0


Validacion OK: no se detectaron conflictos estructurales en el Golden Record.


## Exportacion

El CSV principal queda listo para Feature Engineering. Los archivos auxiliares permiten auditar filas descartadas y conflictos, si existieran.

In [7]:
master.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
discarded.to_csv(DISCARDED_PATH, index=False, encoding="utf-8-sig")
conflicts_alias.to_csv(CONFLICT_ALIAS_PATH, index=False, encoding="utf-8-sig")
conflicts_ruc.to_csv(CONFLICT_RUC_PATH, index=False, encoding="utf-8-sig")

print(f"Archivo maestro exportado: {OUTPUT_PATH}")
print(f"Auditoria de descartados: {DISCARDED_PATH}")
print(f"Conflictos alias/RUC: {CONFLICT_ALIAS_PATH}")
print(f"Conflictos RUC/razon social: {CONFLICT_RUC_PATH}")

Archivo maestro exportado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_ruc_universo_empresas\match_final_empresas_verificado.csv
Auditoria de descartados: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_ruc_universo_empresas\match_final_empresas_verificado_descartados.csv
Conflictos alias/RUC: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_ruc_universo_empresas\match_final_empresas_verificado_conflictos_alias.csv
Conflictos RUC/razon social: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_ruc_universo_empresas\match_final_empresas_verificado_conflictos_ruc.csv
